In [2]:
# Install necessary packages
#%pip install torch torchvision
#%pip install opencv-python
#%pip install pillow

In [1]:
# Import necessary packages

import torch
import cv2
import torch.nn as nn
from PIL import Image
from torchvision import transforms

In [2]:
# Building the CNN model

class CNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()  # Required to run so PyTorch can properly register layers and parameters

    # Extract features from images (nn.Sequential is a container that applies layers in order)
    self.features = nn.Sequential(
        # Convolution layer: 3 input channels (RGB), 16 output channels (16 feature maps)
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        # ReLU activation fn: replaces negatives w/ 0
        # Downsample by a factor of 2 (e.g., (16, 32, 32) --> (16, 16, 16)) & keep strongest features
        nn.ReLU(), nn.MaxPool2d(2),
        # Keep repeating process for other layers; output of previous layer is input to next layer
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2)
    )

    # Turns extracted features into predictions
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(4096, 128),
        nn.ReLU(),
        nn.Dropout(0.4),  # Randomly zero-out 40% of neurons during training (helps prevent overfitting)
        nn.Linear(128, num_classes)
    )

  # How data flows forward through the network
  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [4]:
MODEL_PATH = "./model.pth"  # Get the saved model's path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Creates a device object representing an NVIDIA GPU because we are using the GPU
IMAGE_SIZE = 64
model = CNN(num_classes=7).to(device)

In [5]:
# Build the 'Emotion Detector' video
# Takes the live video --> preprocesses each frame --> runs it through the CNN --> shows prediction on screen

def inference():
  checkpoint = torch.load(MODEL_PATH, map_location=device)  # Load the saved checkpoint from disk
  model.load_state_dict(checkpoint["model_state"])  # Load learned weights into model (restores what the model learned during training)
  model.to(device)  # Moves model to GPU
  model.eval()  # Set model to evaluation mode
  class_names = checkpoint["class_names"]  # Loads label names; used to convert prediction index to a readable label

  print("Webcam is starting...press q to quit")
  webcam = cv2.VideoCapture(0)  # 0 is default camera
  if not webcam.isOpened():
    print("Webcam Error")
    return

  while True:  # Process video continuously
    ret, frame = webcam.read()  # Captures a frame from the webcam

    if not ret:
      break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # OpenCV uses BGR, but PyTorch expects RGB
    resized_image = Image.fromarray(rgb).resize((IMAGE_SIZE, IMAGE_SIZE))  # Resize image to match training size iamges
    tensor = transforms.functional.to_tensor(resized_image)  # Converts image to tensor
    tensor = transforms.functional.normalize(tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Apply same normalization as training so that it's consistent and is able to be classified correctly
    tensor = tensor.unsqueeze(0).to(device)  # Adds batch dimension and moves tensor to GPU

    with torch.no_grad():  # Disable gradient calculation
      output = model(tensor)  # Get outputs from forward pass of CNN
      probs = torch.nn.functional.softmax(output, dim=1)  # Converts outputs to probabilities using softmax activation function
      top_prob, prob_indx = torch.max(probs, dim=1)  # top_prob is the highest probability, prob_indx is the index of the predicted class
      label = class_names[prob_indx.item()]  # Convert the index to a readable label
      confidence = top_prob.item()  # Get the confidence score using the highest probability

    # Output text on the video frame
    cv2.putText(frame, f"{label} {confidence:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.imshow("Emotion Detector (q to quit)", frame)

    # If the user presses 'q', then quit the video
    if cv2.waitKey(1) & 0xFF == ord("q"):
      break

  webcam.release()
  cv2.destroyAllWindows()

In [6]:
# Run program locally
inference()

Webcam is starting...press q to quit
